# LeetCode #95 — Unique Binary Search Trees II: sinh mọi BST bằng chia để trị

**Mức độ:** Medium · **Chủ đề:** Dynamic Programming, Backtracking, Tree, Binary Search Tree, Binary Tree

Skeleton trên LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def generateTrees(self, n: int) -> List[Optional[TreeNode]]:
```

Mục tiêu của notebook:

1. Ôn **cây nhị phân tìm kiếm (BST)**: trái < gốc < phải.
2. Ý tưởng chia để trị: chọn **gốc** `i`, cây trái từ `[lo, i−1]`, cây phải từ `[i+1, hi]`, ghép **mọi cặp**.
3. Memo theo khoảng `(lo, hi)` và chuyện **chia sẻ cây con** (shared subtrees).
4. Liên hệ với **số Catalan** — số cây là `C_n`.

# 1. Bài toán là gì?

Cho số nguyên `n`, trả về **mọi** cây nhị phân tìm kiếm **khác nhau về cấu trúc** có đúng `n` nút với giá trị `1 … n` (mỗi giá trị đúng một lần). Thứ tự tùy ý.

**Constraints (theo đề):** `1 <= n <= 8`

---

## Ví dụ

```text
n = 3  ->  5 cây:

1         1           2          3       3
 \         \         / \        /       /
  3         2       1   3      2       1
 /           \                /         \
2             3              1           2

LeetCode in dạng level-order:
[[1,null,2,null,3],[1,null,3,2],[2,1,3],[3,1,null,null,2],[3,2,null,1]]

n = 1  ->  [[1]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Cây nhị phân tìm kiếm (BST)
3. Đệ quy (recursion)
4. Chia để trị (divide and conquer)

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [7]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [8]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.3 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [9]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [10]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [11]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [12]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [13]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.4 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [14]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [15]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.5 Áp dụng vào bài này

Mọi **BST** chứa `1..n`: chọn từng giá trị `r` làm **gốc**; theo tính chất BST các số `< r` phải ở cây con trái, `> r` ở cây con phải → **chia để trị** sinh mọi cây con trái và phải bằng **đệ quy**, ghép từng cặp. Có thể **ghi nhớ** kết quả theo đoạn `(lo, hi)`.

# 3. Nền tảng: `TreeNode` và cách in cây

In [16]:
from typing import List, Optional
from functools import lru_cache
from collections import deque
import math


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def serialize(root: Optional[TreeNode]) -> list:
    """Level-order kiểu LeetCode, bỏ các None ở cuối."""
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def shape(root: Optional[TreeNode]):
    """Biểu diễn cây dạng tuple lồng nhau (val, trái, phải) — dùng để so sánh."""
    if not root:
        return None
    return (root.val, shape(root.left), shape(root.right))


def inorder(root: Optional[TreeNode]) -> list:
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

t = TreeNode(2, TreeNode(1), TreeNode(3))
print(serialize(t), shape(t), inorder(t))

[2, 1, 3] (2, (1, None, None), (3, None, None)) [1, 2, 3]


## Tính chất BST

Mọi giá trị ở cây con **trái** nhỏ hơn gốc, mọi giá trị ở cây con **phải** lớn hơn gốc, và đúng như vậy với mọi nút. Hệ quả quan trọng: **duyệt inorder (trái–gốc–phải) cho dãy tăng dần**.

Với giá trị `1 … n`: nếu gốc là `i` thì cây trái **bắt buộc** chứa đúng `{1, …, i−1}` và cây phải chứa đúng `{i+1, …, n}`.

# 4. Ý tưởng chính: chọn gốc, ghép mọi cặp cây con

`build(lo, hi)` trả về **danh sách mọi BST** dùng các giá trị `lo … hi`:

```text
nếu lo > hi: return [None]        # một cây rỗng (quan trọng!)
kết quả = []
for i in lo..hi:                   # i là gốc
    for L in build(lo, i - 1):     # mọi cây trái
        for R in build(i + 1, hi): # mọi cây phải
            kết quả.append(TreeNode(i, L, R))
return kết quả
```

`[None]` (danh sách có **một** cây rỗng) chứ không phải `[]` (không có cây nào) — nếu trả `[]`, vòng lặp ghép sẽ không chạy và gốc có một phía rỗng sẽ bị bỏ mất.

In [17]:
def generate_rec(n: int) -> List[Optional[TreeNode]]:
    def build(lo, hi):
        if lo > hi:
            return [None]
        result = []
        for i in range(lo, hi + 1):
            for L in build(lo, i - 1):
                for R in build(i + 1, hi):
                    result.append(TreeNode(i, L, R))
        return result
    return build(1, n)

for tree in generate_rec(3):
    print(serialize(tree))

[1, None, 2, None, 3]
[1, None, 3, 2]
[2, 1, 3]
[3, 1, None, None, 2]
[3, 2, None, 1]


# 5. Mô phỏng: ai làm gốc, mỗi phía có bao nhiêu cây

In [18]:
def trace_generate(n: int):
    @lru_cache(maxsize=None)
    def count(lo, hi):
        if lo > hi:
            return 1
        return sum(count(lo, i - 1) * count(i + 1, hi) for i in range(lo, hi + 1))
    print(f"n = {n}")
    total = 0
    for i in range(1, n + 1):
        left, right = count(1, i - 1), count(i + 1, n)
        total += left * right
        lt = f"{{1..{i-1}}}" if i > 1 else "rỗng"
        rt = f"{{{i+1}..{n}}}" if i < n else "rỗng"
        print(f"  gốc {i}: trái {lt} -> {left} cây, phải {rt} -> {right} cây => {left * right} cây")
    print("  tổng:", total)
    return total

_ = trace_generate(3)
print()
_ = trace_generate(4)

n = 3
  gốc 1: trái rỗng -> 1 cây, phải {2..3} -> 2 cây => 2 cây
  gốc 2: trái {1..1} -> 1 cây, phải {3..3} -> 1 cây => 1 cây
  gốc 3: trái {1..2} -> 2 cây, phải rỗng -> 1 cây => 2 cây
  tổng: 5

n = 4
  gốc 1: trái rỗng -> 1 cây, phải {2..4} -> 5 cây => 5 cây
  gốc 2: trái {1..1} -> 1 cây, phải {3..4} -> 2 cây => 2 cây
  gốc 3: trái {1..2} -> 2 cây, phải {4..4} -> 1 cây => 2 cây
  gốc 4: trái {1..3} -> 5 cây, phải rỗng -> 1 cây => 5 cây
  tổng: 14


# 6. Số Catalan

Số BST với `n` nút là **số Catalan** `C_n`:

```text
C_0 = 1,   C_n = Σ_{i=1..n} C_{i-1} · C_{n-i}   =   (2n)! / ((n+1)! · n!)
```

Đó chính là công thức ở mục 5 (gốc `i`, trái `i−1` nút, phải `n−i` nút). Đây là nội dung của **#96 Unique Binary Search Trees** (chỉ đếm).

In [19]:
catalan = [math.comb(2 * n, n) // (n + 1) for n in range(10)]
print("C_0..C_9 =", catalan)
print("len(generate_rec(n)) =", [len(generate_rec(n)) for n in range(1, 9)])

C_0..C_9 = [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]
len(generate_rec(n)) = [1, 2, 5, 14, 42, 132, 429, 1430]


`C_8 = 1430` — đề giới hạn `n <= 8` vì output tăng **rất nhanh** (khoảng `4^n / n^1.5`).

# 7. Memo theo khoảng — và chia sẻ cây con

`build(lo, hi)` bị gọi nhiều lần với cùng `(lo, hi)`. Memo hóa: mỗi khoảng chỉ dựng một lần, **dùng chung** các cây con.

In [20]:
def generate_memo(n: int) -> List[Optional[TreeNode]]:
    @lru_cache(maxsize=None)
    def build(lo, hi):
        if lo > hi:
            return (None,)
        result = []
        for i in range(lo, hi + 1):
            for L in build(lo, i - 1):
                for R in build(i + 1, hi):
                    result.append(TreeNode(i, L, R))
        return tuple(result)
    return list(build(1, n))

trees = generate_memo(4)
a = next(t for t in trees if t.val == 3 and serialize(t.left) == [1, None, 2])
b = next(t for t in trees if t.val == 4 and t.left.val == 3 and serialize(t.left.left) == [1, None, 2])
print("Cây a:", serialize(a), "| cây b:", serialize(b))
print("Cây con [1, None, 2] của a và của b là cùng một đối tượng?", a.left is b.left.left)

Cây a: [3, 1, 4, None, 2] | cây b: [4, 3, None, 1, None, None, 2]
Cây con [1, None, 2] của a và của b là cùng một đối tượng? True


Hệ quả: nhiều cây trong kết quả **dùng chung nút**. LeetCode chấp nhận điều này (chỉ đọc cây). Nhưng nếu ai đó **sửa** một cây, các cây khác cũng bị đổi theo! Nếu cần các cây độc lập, phải **sao chép sâu** (Bài 4).

Thực ra ngay cả cách đệ quy ở mục 4 (không memo) cũng có chia sẻ: mỗi cây trái `L` được dùng lại cho mọi cây phải `R` trong vòng lặp. Memo chỉ làm việc chia sẻ nhiều hơn. Muốn các cây hoàn toàn độc lập thì phải sao chép (Bài 4).

# 8. Phiên bản đúng format LeetCode

In [21]:
class Solution:
    def generateTrees(self, n: int) -> List[Optional[TreeNode]]:
        @lru_cache(maxsize=None)
        def build(lo: int, hi: int):
            if lo > hi:
                return (None,)
            result = []
            for i in range(lo, hi + 1):
                for L in build(lo, i - 1):
                    for R in build(i + 1, hi):
                        result.append(TreeNode(i, L, R))
            return tuple(result)

        return list(build(1, n))

In [22]:
import itertools

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


def is_bst(root, lo=float("-inf"), hi=float("inf")):
    if not root:
        return True
    return lo < root.val < hi and is_bst(root.left, lo, root.val) and is_bst(root.right, root.val, hi)


def bst_insert(root, v):
    if not root:
        return TreeNode(v)
    if v < root.val:
        root.left = bst_insert(root.left, v)
    else:
        root.right = bst_insert(root.right, v)
    return root


sol = Solution()
check("Ví dụ 1", sorted(serialize(t) for t in sol.generateTrees(3)),
      sorted([[1, None, 2, None, 3], [1, None, 3, 2], [2, 1, 3], [3, 1, None, None, 2], [3, 2, None, 1]], key=str))
check("Ví dụ 2", [serialize(t) for t in sol.generateTrees(1)], [[1]])

for n in range(1, 9):
    trees = sol.generateTrees(n)
    shapes = {shape(t) for t in trees}
    assert len(trees) == len(shapes) == catalan[n]                       # đủ và không trùng
    assert all(is_bst(t) and inorder(t) == list(range(1, n + 1)) for t in trees)
for n in range(1, 7):
    # brute force: mọi thứ tự chèn tạo ra đúng tập cấu trúc BST
    brute = set()
    for perm in itertools.permutations(range(1, n + 1)):
        root = None
        for v in perm:
            root = bst_insert(root, v)
        brute.add(shape(root))
    assert {shape(t) for t in sol.generateTrees(n)} == brute
print("n = 1..8: đúng C_n cây, không trùng, đều là BST; n <= 6 khớp brute force qua mọi thứ tự chèn")

Ví dụ 1      PASS  [[1, None, 2, None, 3], [1, None, 3, 2], [2, 1, 3], [3, 1, None, None, 2], [3, 2, None, 1]]
Ví dụ 2      PASS  [[1]]
n = 1..8: đúng C_n cây, không trùng, đều là BST; n <= 6 khớp brute force qua mọi thứ tự chèn


# 9. Vì sao đúng? Độ phức tạp?

- Mỗi BST trên `{lo … hi}` có **một** gốc `i`; cây trái là BST trên `{lo … i−1}`, cây phải là BST trên `{i+1 … hi}`. Ngược lại, mọi cách ghép như vậy đều tạo ra một BST hợp lệ.
- Hai cây khác gốc thì khác nhau; cùng gốc thì khác nhau ⇔ khác cây trái hoặc khác cây phải → không trùng.

```text
Số cây:  C_n ≈ 4^n / (n^1.5 · √π)
Time:    O(n · C_n) để dựng output (không memo còn chậm hơn)
Space:   O(n · C_n) cho output; memo chia sẻ cây con nên ít nút thực tế hơn
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Trường hợp gốc trả `[]` thay vì `[None]`

In [23]:
def wrong_empty_list(n: int) -> List[Optional[TreeNode]]:
    def build(lo, hi):
        if lo > hi:
            return []                               # SAI: không có cây nào -> vòng ghép không chạy
        result = []
        for i in range(lo, hi + 1):
            for L in build(lo, i - 1):
                for R in build(i + 1, hi):
                    result.append(TreeNode(i, L, R))
        return result
    return build(1, n)

print("Sai : n = 3 ->", len(wrong_empty_list(3)), "cây | Đúng:", len(sol.generateTrees(3)))

Sai : n = 3 -> 0 cây | Đúng: 5


Không cây nào có phía rỗng được tạo ra — mà **lá** luôn có cả hai phía rỗng → kết quả rỗng.

## Lỗi 2: Dùng chung một nút gốc cho nhiều cây

In [24]:
def wrong_shared_root(n: int) -> List[Optional[TreeNode]]:
    def build(lo, hi):
        if lo > hi:
            return [None]
        result = []
        for i in range(lo, hi + 1):
            root = TreeNode(i)                      # SAI: tạo một lần, dùng cho mọi cặp (L, R)
            for L in build(lo, i - 1):
                for R in build(i + 1, hi):
                    root.left, root.right = L, R
                    result.append(root)
        return result
    return build(1, n)

print("Sai :", [serialize(t) for t in wrong_shared_root(3)])

Sai : [[1, None, 3, 2], [1, None, 3, 2], [2, 1, 3], [3, 2, None, 1], [3, 2, None, 1]]


Các phần tử của `result` là **cùng một** đối tượng, nên đều mang cặp `(L, R)` cuối cùng được gán. Phải tạo `TreeNode(i, L, R)` **mới** cho mỗi cặp.

## Lỗi 3: Nhầm "khác cấu trúc" với "khác thứ tự chèn"

Có `n!` thứ tự chèn nhưng chỉ `C_n` cấu trúc (vd. chèn `2,1,3` và `2,3,1` cho cùng một cây). Sinh qua hoán vị rồi khử trùng là cách brute force (dùng để kiểm tra ở mục 8), không phải lời giải.

## Lỗi 4: Trả `[[]]` hoặc `[None]` khi `n = 0`

Phiên bản cũ của đề có `n = 0` và đáp án là `[]`; phiên bản hiện tại có `n >= 1`. Nếu tự mở rộng, nhớ xử lý riêng.

# 11. Mẹo nhận diện pattern

**"Sinh mọi cấu trúc cây / biểu thức"** → chia để trị: chọn **điểm chia** (gốc, toán tử), đệ quy hai phía, ghép **tích Descartes** các kết quả.

Bài liên quan: **#96 Unique BSTs** (đếm — Catalan), **#241 Different Ways to Add Parentheses** (chọn toán tử cuối cùng), **#894 All Possible Full Binary Trees**, **#22 Generate Parentheses** (cũng đếm bằng Catalan).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Có bao nhiêu BST với `n = 4` có **gốc là 2**? Vẽ chúng.

## Bài 2 — Điền chỗ trống

In [25]:
def exercise_2_generate_trees(n: int) -> List[Optional[TreeNode]]:
    def build(lo, hi):
        # TODO: trường hợp rỗng trả [None]; chọn gốc i; ghép mọi (L, R)
        pass
    return build(1, n)


# assert sorted(map(str, map(serialize, exercise_2_generate_trees(3)))) == sorted(map(str, [[1, None, 2, None, 3], [1, None, 3, 2], [2, 1, 3], [3, 1, None, None, 2], [3, 2, None, 1]]))
# assert len(exercise_2_generate_trees(5)) == 42

## Bài 3 — Biến thể: Unique Binary Search Trees (LeetCode #96)

Chỉ **đếm** số BST với `n` nút (`1 <= n <= 19`), bằng DP `O(n²)`.

In [26]:
def exercise_3_num_trees(n: int) -> int:
    # TODO
    pass


# assert exercise_3_num_trees(3) == 5
# assert exercise_3_num_trees(1) == 1
# assert exercise_3_num_trees(19) == 1767263190

## Bài 4 — Sao chép sâu để các cây độc lập

Viết `clone(root)` trả về bản sao sâu của cây, rồi `generate_independent(n)` trả về các cây **không chia sẻ nút**.

In [27]:
def exercise_4_generate_independent(n: int) -> List[Optional[TreeNode]]:
    # TODO
    pass


# ts = exercise_4_generate_independent(4); ids = [id(x) for t in ts for x in [t] + [None]]; assert len(ts) == 14
# ts = exercise_4_generate_independent(4); nodes = []; stack = list(ts)
# while stack: x = stack.pop(); stack += [c for c in (x.left, x.right) if c] if x else []; nodes.append(x)
# assert len(nodes) == len({id(x) for x in nodes}) == 14 * 4

## Bài 5 — Mini challenge: All Possible Full Binary Trees (LeetCode #894)

Cây nhị phân **đầy đủ**: mỗi nút có 0 hoặc 2 con. Trả về mọi cây đầy đủ có đúng `n` nút (mọi nút có giá trị 0).

```python
len(all_fbt(7)) -> 5
all_fbt(2)      -> []      # n chẵn thì không có
```

In [28]:
def exercise_5_all_fbt(n: int) -> List[Optional[TreeNode]]:
    # TODO
    pass


# assert len(exercise_5_all_fbt(7)) == 5
# assert exercise_5_all_fbt(2) == []
# assert [serialize(t) for t in exercise_5_all_fbt(3)] == [[0, 0, 0]]
# assert len(exercise_5_all_fbt(9)) == 14

# 13. Gợi ý

**Bài 1**
- Mức 1: gốc 2 → trái `{1}` (1 cây), phải `{3, 4}` (2 cây) → 2 cây.

**Bài 3**
- Mức 1: `G = [1] + [0] * n`; `G[k] = Σ_{i=1..k} G[i-1] · G[k-i]`.

**Bài 4**
- Mức 1: `clone(None) = None`; `clone(t) = TreeNode(t.val, clone(t.left), clone(t.right))`.

**Bài 5**
- Mức 1: `n` chẵn → `[]`; `n == 1` → `[TreeNode(0)]`.
- Mức 2: chia `n − 1` nút còn lại thành `left` (lẻ) và `n − 1 − left` (lẻ), ghép như bài chính.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [29]:
for t in sol.generateTrees(4):
    if t.val == 2:
        print(serialize(t))

[2, 1, 3, None, None, None, 4]
[2, 1, 4, None, None, 3]


## Đáp án Bài 2

In [30]:
def answer_2_generate_trees(n: int) -> List[Optional[TreeNode]]:
    return generate_rec(n)

assert sorted(map(str, map(serialize, answer_2_generate_trees(3)))) == sorted(map(str, [[1, None, 2, None, 3], [1, None, 3, 2], [2, 1, 3], [3, 1, None, None, 2], [3, 2, None, 1]]))
assert len(answer_2_generate_trees(5)) == 42
print("Bài 2 PASS (giống generate_rec ở mục 4)")

Bài 2 PASS (giống generate_rec ở mục 4)


## Đáp án Bài 3

In [31]:
def answer_3_num_trees(n: int) -> int:
    G = [1] + [0] * n
    for k in range(1, n + 1):
        G[k] = sum(G[i - 1] * G[k - i] for i in range(1, k + 1))
    return G[n]

assert answer_3_num_trees(3) == 5
assert answer_3_num_trees(1) == 1
assert answer_3_num_trees(19) == 1767263190
assert all(answer_3_num_trees(n) == catalan[n] for n in range(10))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [32]:
def clone(t: Optional[TreeNode]) -> Optional[TreeNode]:
    return TreeNode(t.val, clone(t.left), clone(t.right)) if t else None

def answer_4_generate_independent(n: int) -> List[Optional[TreeNode]]:
    return [clone(t) for t in sol.generateTrees(n)]

ts = answer_4_generate_independent(4); assert len(ts) == 14
nodes = []; stack = list(ts)
while stack:
    x = stack.pop()
    if x:
        stack += [c for c in (x.left, x.right) if c]
        nodes.append(x)
assert len(nodes) == len({id(x) for x in nodes}) == 14 * 4
shared = generate_memo(4)
all_nodes = []; stack = list(shared)
while stack:
    x = stack.pop()
    if x:
        stack += [x.left, x.right]
        all_nodes.append(id(x))
print("Bài 4 PASS | bản memo: tổng", len(all_nodes), "lượt nút nhưng chỉ", len(set(all_nodes)), "nút thật (chia sẻ)")

Bài 4 PASS | bản memo: tổng 56 lượt nút nhưng chỉ 34 nút thật (chia sẻ)


## Đáp án Bài 5

In [33]:
def answer_5_all_fbt(n: int) -> List[Optional[TreeNode]]:
    @lru_cache(maxsize=None)
    def build(k):
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(0),)
        out = []
        for left in range(1, k - 1, 2):
            for L in build(left):
                for R in build(k - 1 - left):
                    out.append(TreeNode(0, L, R))
        return tuple(out)
    return list(build(n))

assert len(answer_5_all_fbt(7)) == 5
assert answer_5_all_fbt(2) == []
assert [serialize(t) for t in answer_5_all_fbt(3)] == [[0, 0, 0]]
assert len(answer_5_all_fbt(9)) == 14
print("Bài 5 PASS | số cây đầy đủ với n = 1, 3, 5, …, 15:", [len(answer_5_all_fbt(k)) for k in range(1, 16, 2)])

Bài 5 PASS | số cây đầy đủ với n = 1, 3, 5, …, 15: [1, 1, 2, 5, 14, 42, 132, 429]


Số cây nhị phân đầy đủ với `2m + 1` nút cũng là `C_m` — Catalan xuất hiện khắp nơi trong các bài "đếm cấu trúc cây".

# 15. Tổng kết

> `build(lo, hi)`: với mỗi gốc `i`, ghép mọi cây trái từ `build(lo, i−1)` với mọi cây phải từ `build(i+1, hi)`; khoảng rỗng trả `[None]`. Số cây là Catalan `C_n`. Memo theo `(lo, hi)` giúp chia sẻ cây con.

---

## Checklist trước khi nộp

- [ ] Khoảng rỗng trả `[None]`, không phải `[]`?
- [ ] Tạo `TreeNode(i, L, R)` **mới** cho mỗi cặp?
- [ ] Hiểu rằng bản memo chia sẻ cây con?
- [ ] Số cây bằng `C_n` (kiểm tra nhanh)?